In [1]:
# pip install shared_utils 

In [2]:
# %pip install matplotlib-scalebar

In [3]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
from segment_speed_utils.project_vars import PUBLIC_GCS
db_engine = get_engine()
credentials, project = google.auth.default()
fs = gcsfs.GCSFileSystem()
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.patches import Patch
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

from matplotlib.lines import Line2D
from matplotlib_scalebar.scalebar import ScaleBar

pd.set_option('display.max_columns', None)
import numpy as np

In [4]:
df_90 = gpd.read_parquet("ca_top90pct_density_dissolved.parquet")

In [5]:
df_90.head(5)

,geometry,n_block_groups,total_population,total_land_area_sqmi
0,"MULTIPOLYGON (((-120.71444 37.39681, -120.7230...",22506,35357649.0,6614.833217


In [6]:
import matplotlib.patches as Patches

In [7]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses'

In [8]:
# Load the stored organization dataset from the specified GCS file path.
with fs.open(f"{GCS_FILE_PATH}/transit_provider_dashboard/organization_stops_buffered_route_array.parquet", "rb") as f:
    orgs_stop_buffered_route = gpd.read_parquet(f)

In [9]:
# Load the stored organization dataset from the specified GCS file path.
with fs.open(f"{GCS_FILE_PATH}/transit_provider_dashboard/organization_stops_buffered.parquet", "rb") as f:
    orgs_stop_buffered = gpd.read_parquet(f)

In [10]:
# Load the stored ACS dataset from the specified GCS file path.
with fs.open(f"{GCS_FILE_PATH}/transit_provider_dashboard/census_tracts_data_2024.parquet", "rb") as f:
    tracts_ca_acs = gpd.read_parquet(f)

In [11]:
# Load Ridership Grouped Data 
with fs.open(f"{GCS_FILE_PATH}/transit_provider_dashboard/ridership_data_2024.parquet", "rb") as f:
    ridership_data_grouped = pd.read_parquet(f)

In [12]:
# Load Ridership Grouped Data 
with fs.open(f"{GCS_FILE_PATH}/transit_provider_dashboard/route_id_shapes.parquet", "rb") as f:
    route_id_shapes = gpd.read_parquet(f)

In [13]:
tracts_ca_acs.crs

<Projected CRS: EPSG:3310>
Name: NAD83 / California Albers
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: United States (USA) - California.
- bounds: (-124.45, 32.53, -114.12, 42.01)
Coordinate Operation:
- name: California Albers
- method: Albers Equal Area
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [14]:
orgs_stop_buffered.crs

<Projected CRS: EPSG:3310>
Name: NAD83 / California Albers
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: United States (USA) - California.
- bounds: (-124.45, 32.53, -114.12, 42.01)
Coordinate Operation:
- name: California Albers
- method: Albers Equal Area
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [15]:
orgs_stop_buffered.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 133556 entries, 0 to 133555
Data columns (total 16 columns):
 #   Column                         Non-Null Count   Dtype   
---  ------                         --------------   -----   
 0   name                           133556 non-null  object  
 1   ntd_id_x                       85055 non-null   object  
 2   ntd_id_2022_x                  85312 non-null   object  
 3   stop_id                        133556 non-null  object  
 4   stop_name                      133556 non-null  object  
 5   schedule_gtfs_dataset_name     96325 non-null   object  
 6   organization_source_record_id  97614 non-null   object  
 7   geometry                       133556 non-null  geometry
 8   analysis_name                  97695 non-null   object  
 9   organization_name              96325 non-null   object  
 10  name_clean                     133556 non-null  object  
 11  source_record_id               68069 non-null   object  
 12  key     

In [16]:
orgs_stop_buffered.analysis_name.nunique()

192

In [17]:
orgs_stop_buffered_route.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 112889 entries, 0 to 112888
Data columns (total 18 columns):
 #   Column                         Non-Null Count   Dtype   
---  ------                         --------------   -----   
 0   name                           112889 non-null  object  
 1   ntd_id_x                       85191 non-null   object  
 2   ntd_id_2022_x                  85445 non-null   object  
 3   stop_id                        112889 non-null  object  
 4   stop_name                      112889 non-null  object  
 5   route_id_array                 112889 non-null  object  
 6   feed_key                       94213 non-null   object  
 7   schedule_gtfs_dataset_name     96489 non-null   object  
 8   organization_source_record_id  97649 non-null   object  
 9   geometry                       112889 non-null  geometry
 10  analysis_name                  97877 non-null   object  
 11  organization_name              96489 non-null   object  
 12  name_cle

In [18]:
route_id_shapes.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 11219 entries, 0 to 11218
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   feed_key  11219 non-null  object  
 1   route_id  11219 non-null  object  
 2   shape_id  11219 non-null  object  
 3   geometry  11219 non-null  geometry
dtypes: geometry(1), object(3)
memory usage: 350.7+ KB


In [19]:
org_feeds = set(orgs_stop_buffered_route["feed_key"].dropna().astype(str).str.strip())
shape_feeds = set(route_id_shapes["feed_key"].dropna().astype(str).str.strip())

print(f"Org feed_keys: {len(org_feeds)}")
print(f"Shape feed_keys: {len(shape_feeds)}")

print("\nMatching feed_keys:")
print(sorted(org_feeds & shape_feeds))

Org feed_keys: 177
Shape feed_keys: 226

Matching feed_keys:
['0195f5a2056498103ada733ba3484f21', '020fff35c33cc2d4501bf7674789e83e', '0905881fcd925e6f8438f5362c98e2b6', '10e12c5a165b5afa57f3fa110d619b01', '1324dbf6ab954e18b00ed0c18798a4c5', '149c7110209ee36b18eb8d0b7bca7b1a', '14bd614cc4b55ad1a924f759da2cf35f', '16349880d0dd18cb136625e264ab7f16', '167479e89da79dd6a272c1cca04452d8', '186750f1308db1f41f43fe7d661981ba', '1977a6588484376657d473b2359f572c', '1a97d22c9a60525d9501bde205b99f16', '1acac64270bf0ac8f4c5a44076237ca3', '1b06e2f1dbde2e2da76c959139d13fa7', '1b198910a814f502e4417b5531720db2', '1ba6bd2aa774158205e2c50342506391', '1cda94c39690a3708c747e57706351b4', '1dc19b0c7c704e7663492625b4c1acb1', '1f6bdbd2bd5e2e25c07750c659df6c0b', '214b0653d67c00b17c05de7cf1a9fe88', '21ab48d0ff065bf6ba1fa1b100ca6c29', '2243361a94f531aebfc3238755efafe0', '23d1893801eefadf7544a670a3bcd312', '263b78e9c2dc542a1eb770b55e0af8d5', '2a72b61f9696d01c554615a5ea71dcde', '2ab000f3b8c75a3b5afd6b290b4e1e03', '2

In [20]:
allowed_vctc_intercity = [
    "Thousand Oaks Transit Center", "The Oaks", "Camarillo Metrolink Station", "Carmen Plaza",
    "Camarillo Outlets Food Court", "Esplanade Mall NB", "Ventura Pier",
    "Pacific View Mall (Ventura Transit Center)", "Esplanade Mall SB",
    "Camarillo Outlets - Main Court", "Thousand Oaks Library / Teen Center",
    "Moorpark Station EB", "Princeton Ave at Amherst NB", "Simi Town Center EB",
    "C Street Transfer Center NB", "Oxnard College EB", "CSU Channel Islands",
    "Villa Calleguas", "Oxnard College WB", "Fillmore Active Adult & Community Center",
    "Santa Paula City Hall", "Santa Paula DMV (Harvard & Steckel)",
    "Santa Paula Kmart", "Ventura College (NW corner of Telegraph/Estates Ave)",
    "Ventura College (SW Corner of Telegraph/Estates Ave)",
    "Santa Paula DMV (Harvard at Craig)", "Moorpark Station WB",
    "Santa Clara and Oak WB (Downtown Ventura)", "Carpinteria Ave at Eugenia Pl (Downtown Carp)",
    "Cabrillo at Puerta Vallarta WB (East Beach)", "Gutierrez at Garden",
    "Figueroa at Chapala (MTD Transit Center)", "Bath and Pueblo NB (Cottage Hospital)",
    "SB Airport NB", "UCSB Bus Loop", "SB Airport SB", "Haley at Garden",
    "Cabrillo at Puerta Vallarta EB (East Beach)", "Carpinteria Ave at Maple (Downtown Carp)",
    "Santa Clara and Oak EB (Downtown Ventura)", "Princeton Ave at Amherst SB",
    "Gov Center (NE Telephone/Victoria)", "Buena High School (NW Telegraph/Wake Forest)",
    "VCMC Westbound", "St. Bon HS Westbound", "Saint Bonaventure High School",
    "Ventura County Medical Center", "Buena High School",
    "Gov. Center (SE Telephone/Victoria)", "Ventura County Government Center (Hill Rd/Thille St)",
    "West Main at Peking WB (Park & Ride)", "Camino del Remedio WB (SB County Complex)",
    "Turnpike & San Gordiano (SMHS)", "Hollister and Patterson WB (Goleta College Hospital)",
    "Hollister and Kellogg WB", "Hollister and Nectarine", "Hollister and Pine",
    "Hollister and Kellogg EB", "West Main at Peking EB (Park & Ride)",
    "Ventura County Government Center (NW Victoria Ave/Telephone Rd)",
    "NW Cochran St/Galena St (Simi Valley Park & Ride)", "Simi Town Center WB",
    "Simi Valley Metrolink", "Simi Valley Civic Center - SE",
    "Somis SW corner Somis Rd/Rice St", "Ventura County Government Center (NE Victoria Ave/Telephone Rd)",
    "Somis NE corner Somis Rd/Rice St", "SW Cochran St/Galena St",
    "Simi Valley Civic Center - SW", "Santa Barbara at De La Guerra",
    "Figueroa at Santa Barbara St (Courthouse)", "De La Vina & Mission",
    "Anacapa at Anapamu (SB Library)", "Anacapa at De La Guerra (SB City Hall)",
    "Victory & Topanga", "Victory & De Soto", "Victory & Mason", "Burbank & De Soto",
    "Thousand Oaks Transit Center NB", "Hollister & Palo Alto SB", "Hollister & Entrance SB",
    "Cortona at Castilian", "Castilian at Los Carneros", "Los Carneros Rd/Karl Storz Way NB",
    "Hollister and Aero Camino EB", "Hollister & La Patera SB", "Hollister & La Patera NB",
    "Hollister and Aero Camino WB", "Los Carneros Rd/Karl Storz Way SB",
    "Castilian at Los Carneros NB", "Castilian at Cortona", "Hollister & Entrance NB",
    "Hollister & Palo Alto NB", "Hollister and Patterson EB (Goleta Cottage Hospital)",
    "San Marcos High School"
]

allowed_valley_express_stops = [
    "Fillmore Terminal Inbound", "Fillmore Terminal Outbound", "7-Eleven", "Mercado La Plaza",
    "Faith Community Church", "C St. & Sespe Ave.", "C St. & Meadowlark Drive", "Shiell Park",
    "Mountain Vista School", "4th & B St", "4th & B St (Eastbound)", "Third St. & B St.",
    "City Hall", "Downtown", "Fillmore MS & HS", "Central Ave & 4th St.", "San Cayetano School",
    "Mountain View St. & Third St.", "Los Serenos Drive & Sierra Vista Ave", "Los Serenos Drive",
    "El Paso St. & Sierra Vista Ave.", "Sierra Vista Ave & Sespe Ave", "Sespe Ave. & Burson Ln.",
    "Sespe Ave. & Burson Ln. (Westbound)", "Price St. & First St.", "McCampbell St. & Wileman St.",
    "B St. & Santa Clara St.", "B St. & Santa Clara St. (Northbound)", "River St. & Deerfield Drive",
    "River St. & Deerfield Drive (Eastbound)", "Santa Fe St. & Rio Grande St.", "Surrey Way & Santa Fe St.",
    "Santa Fe St. & Reading St.", "River St. (Westbound)", "Rio Vista Elementary", "Boys & Girls Club",
    "El Dorado Estate", "Rancho Sespe", "Main St. & Shannon Ln (park)", "Piru Square", "Valle Naranjal",
    "Santa Barbara St. & 4th St.", "Santa Barbara St. & 11th St.", "12th St. & Santa Paula St.",
    "Main St. & 12th St.", "Harvard Blvd. & Garcia St.", "Harvard Blvd. & Ojai St.",
    "Harvard Blvd. & Steckel Drive", "Harvard Blvd. & Laurie Ln.", "Harvard Blvd. & Craig Drive",
    "Harvard Blvd. & 8th St.", "10th St. & Santa Barbara St.", "Main St. & 11th St.",
    "Main St. & 4th St.", "Main St. & Mill St.", "10th St. & Virginia Terr.", "Santa Barbara St. & 8th St.",
    "Santa Paula Hospital", "Beckwith Rd. & Via Pasada", "Santa Paula St. & Walden St.",
    "12th St. & Richmond Rd.", "Barbara Webster Elem.", "Bedell School", "McKevett School",
    "Santa Paula High School", "Isbell School", "Glen City School", "Blanchard School",
    "Moorpark College", "Moorpark Metrolink WB", "Moorpark Metrolink EB"
]


allowed_city_of_camarillo_stops = [
    "Leisure Village Club House",
    "East Gate/ Village #44",
    "Santa Rosa Plaza",
    "Plaza at Mission Oaks (Pardee Plaza)",
    "Camarillo Library",
    "Pleasant Valley Hospital",
    "Village Square",
    "Central Plaza",
    "Ponderosa Plaza",
    "Post Office",
    "Community Center",
    "Mira Vista Village",
    "Pleasant Valley Park",
    "Mission Oaks Plaza (Inbound)",
    "Mission Oaks Plaza (Outbound)",
    "East Gate/Village #41",
    "Mtn. View/ Village #23",
    "Mtn. View/Village #24",
    "Mtn. View /Village #25",
    "Mtn. View/ #30",
    "Mtn. View/ Village #4",
    "Mtn. View/Village #16"
]

allowed_city_of_ojai_stops = [
    "Ojai Ave @ Arcade",
    "Ojai Ave & Signal",
    "Ojai Ave @ Westridge Mkt",
    "Rice & Camille",
    "Rice & Fierro",
    "Rice & El Sereno Estates",
    "Rice & Woodland",
    "Rice & Hwy 150 NW Corner",
    "Woodland & Hwy 33",
    "Hwy 33 @ Red Horse Plaza",
    "La Luna @ Fire Station",
    "El Roblar & La Luna",
    "El Roblar & Encinal",
    "El Roblar & Lomita",
    "El Roblar @ Roth Apts",
    "Hospital",
    "Y Memorial Garden",
    "Loma @ Mira Valle MH Park",
    "Ojai Valley Inn",
    "Ojai @ Bank of America",
    "Nordhoff HS"
]


# Identify rows currently assigned to the broad VCTC group.
vctc_parent = (
    "Ventura County (VCTC, Gold Coast, Cities of Camarillo, "
    "Moorpark, Ojai, Simi Valley, Thousand Oaks)"
)
mask_vctc = orgs_stop_buffered["analysis_name"].eq(vctc_parent)

# Map each VCTC subgroup to its allowed stops.
vctc_groups = {
    "VCTC Intercity": allowed_vctc_intercity,
    "City of Camarillo": allowed_city_of_camarillo_stops,
    "City of Ojai": allowed_city_of_ojai_stops,
    "VCTC Valley Express": allowed_valley_express_stops,
}

# Reassign VCTC stops to their specific subgroup.
for group, stops in vctc_groups.items():
    orgs_stop_buffered.loc[
        mask_vctc & orgs_stop_buffered["stop_name"].isin(stops),
        "analysis_name"
    ] = group

# Store NTD and operating statistics by agency.
agency_map_ntd = {
    "City of Ojai": {"ntd_id_2022": "91058", "upt": 48315, "voms": 2},
    "City of Camarillo": {"ntd_id_2022": "90163", "upt": 73504, "voms": 18},
}

# for agency, vals in orgs_stop_buffered.items():
#     mask = orgs_stop_buffered["analysis_name"].eq(agency)

#     orgs_stop_buffered.loc[mask, "ntd_id_2022_y"] = vals["ntd_id_2022_y"]
#     orgs_stop_buffered.loc[mask, "unlinked_passenger_trips_upt"] = vals["upt"]
#     orgs_stop_buffered.loc[mask, "agency_voms"] = vals["voms"]



In [21]:
# Define stops that belong to Metrolink.
allowed_metrolink_stops = [
    "L.A. Union Station", "Cal State LA", "El Monte", "Baldwin Park",
    "Covina", "Pomona - North", "Claremont", "Montclair", "Upland",
    "Rancho Cucamonga", "Fontana", "Rialto", "San Bernardino Depot",
    "San Bernardino - Downtown", "Redlands - University",
    "Redlands - Downtown", "Redlands - Esri", "San Bernardino - Tippecanoe"
]

# Identify rows currently assigned to the Metrolink agency.
metrolink_mask = orgs_stop_buffered["analysis_name"].eq(
    "Southern California Regional Rail Authority"
)

# Keep all other agencies and only allowed stops for Metrolink.
orgs_stop_buffered = orgs_stop_buffered[
    ~metrolink_mask | orgs_stop_buffered["stop_name"].isin(allowed_metrolink_stops)
].copy()

In [22]:
# Reconciliation groups for different organization subset
RECONCILIATION_GROUPS = {
    "contactless_credit_debit": [
        "Santa Cruz Metro",
        "City of Santa Monica",
        "Palos Verdes Peninsula Transit Authority",
        "Los Angeles World Airports",
        "Los Angeles County Metropolitan Transportation Authority",
        "Los Angeles County",
        "Long Beach Transit",
        "City of Los Angeles",
        "City of Glendale",
        "Foothill Transit",
        "City of Torrance",
        "City of Santa Clarita",
        "City of Redondo Beach",
        "City of Pasadena",
        "City of Norwalk",
        "City of Monterey Park",
        "City of Montebello",
        "City of Lawndale",
        "City of Huntington Park",
        "City of Glendora",
        "City of Gardena",
        "City of Culver City",
        "City of Compton",
        "City of Carson",
        "City of Burbank",
        "City of Baldwin Park",
        "Antelope Valley Transit Authority",
        "Western Contra Costa Transit Authority",
        "Sonoma-Marin Area Rail Transit District",
        "Sonoma County Transit Schedule",
        "Solano Transportation Authority",
        "Santa Clara Valley Transportation Authority",
        "San Mateo County Transit District",
        "San Francisco Bay Ferry and Oakland Alameda Water Shuttle Schedule",
        "San Francisco Bay Area Rapid Transit District",
        "City of Petaluma",
        "Napa Valley Transportation Authority",
        "Marin County Transit District",
        "Livermore-Amador Valley Transit Authority",
        "Golden Gate Bridge",
        "Eastern Contra Costa Transit Authority",
        "Madera County",
        "Cloverdale Transit",
        "City of Vacaville",
        "City of Union City",
        "City of Santa Rosa",
        "City of Fairfield",
        "SF Muni",
        "Peninsula Corridor Joint Powers Board",
        "Alameda-Contra Costa Transit District",
        "Dumbarton Bridge Regional Operations Consortium",
        "Orange County Transportation Authority",
        "North County Transit District",
        "San Diego Metropolitan Transit System, Airport, Flagship Cruises",
        "Anaheim Transportation Network",
        "Capitol Corridor Joint Powers Authority",
        "City of Camarillo",
        "City of Moorpark",
        "City of Morro Bay",
        "Redding Area Bus Authority",
        "City of Simi Valley",
        "City of San Luis Obispo",
        "City of Thousand Oaks",
        "El Dorado County Transit Authority",
        "Gold Coast Transit District",
        "Humboldt Transit Authority",
        "Lake Transit Authority",
        "Mendocino Transit Authority",
        "Southern California Regional Rail Authority",
        "Monterey-Salinas Transit",
        "Nevada County",
        "Redwood Coast Transit Authority",
        "Sacramento Regional Transit District",
        "Santa Cruz Metro",
        "Santa Barbara County Association of Governments",
        "Santa Barbara Metropolitan Transit District",
        "San Luis Obispo Regional Transit Authority",
        "VCTC Valley Express",
        "Ventura County (VCTC, Gold Coast, Cities of Camarillo, Moorpark, Ojai, Simi Valley, Thousand Oaks)",
        "Yolo County Transportation District"
    ],

    "contactless_payments_now_ca_msas": [
        "Anaheim Transportation Network",
        "Capitol Corridor Joint Powers Authority",
        "City of Camarillo",
        "City of Moorpark",
        "City of Morro Bay",
        "Redding Area Bus Authority",
        "City of Simi Valley",
        "City of San Luis Obispo",
        "City of Thousand Oaks",
        "El Dorado County Transit Authority",
        "Gold Coast Transit District",
        "Humboldt Transit Authority",
        "Lake Transit Authority",
        "Mendocino Transit Authority",
        "Southern California Regional Rail Authority",
        "Monterey-Salinas Transit",
        "Nevada County",
        "Redwood Coast Transit Authority",
        "Sacramento Regional Transit District",
        "Santa Cruz Metro",
        "Santa Barbara County Association of Governments",
        "Santa Barbara Metropolitan Transit District",
        "San Luis Obispo Regional Transit Authority",
        "VCTC Valley Express",
        "Ventura County (VCTC, Gold Coast, Cities of Camarillo, Moorpark, Ojai, Simi Valley, Thousand Oaks)",
        "Yolo County Transportation District"
    ],

    "contactless_q2": [
        "Alameda-Contra Costa Transit District",
        "Anaheim Transportation Network",
        "Antelope Valley Transit Authority",
        "Peninsula Corridor Joint Powers Board",
        "Capitol Corridor Joint Powers Authority",
        "City and County of San Francisco",
        "City of Baldwin Park",
        "City of Burbank",
        "City of Carson",
        "City of Compton",
        "City of Culver City",
        "City of Fairfield",
        "City of Gardena",
        "City of Glendora",
        "City of Huntington Park",
        "City of Lawndale",
        "City of Montebello",
        "City of Monterey Park",
        "City of Morro Bay",
        "City of Norwalk",
        "City of Pasadena",
        "Redding Area Bus Authority",
        "City of Redondo Beach",
        "City of Santa Clarita",
        "City of Santa Rosa",
        "City of San Luis Obispo",
        "City of Torrance",
        "City of Union City",
        "City of Vacaville",
        "Cloverdale Transit",
        "Central Contra Costa Transit Authority",
        "Eastern Contra Costa Transit Authority",
        "El Dorado County Transit Authority",
        "Foothill Transit",
        "City of Glendale",
        "Golden Gate Bridge",
        "Humboldt Transit Authority",
        "City of Los Angeles",
        "Lake Transit Authority",
        "Livermore-Amador Valley Transit Authority",
        "Los Angeles County",
        "Los Angeles County Metropolitan Transportation Authority",
        "Los Angeles World Airports",
        "Marin County Transit District",
        "Mendocino Transit Authority",
        "Monterey-Salinas Transit",
        "Napa Valley Transportation Authority",
        "North County Transit District",
        "Nevada County",
        "Orange County Transportation Authority",
        "Palos Verdes Peninsula Transit Authority",
        "Dumbarton Bridge Regional Operations Consortium",
        "City of Petaluma",
        "Redwood Coast Transit Authority",
        "Sacramento Regional Transit District",
        "San Diego Metropolitan Transit System, Airport, Flagship Cruises",
        "San Francisco Bay Area Rapid Transit District",
        "San Mateo County Transit District",
        "Santa Clara Valley Transportation Authority",
        "City of Santa Monica",
        "Santa Barbara County Association of Governments",
        "Santa Barbara Metropolitan Transit District",
        "San Luis Obispo Regional Transit Authority",
        "Solano Transportation Authority",
        "Sonoma County",
        "Sonoma-Marin Area Rail Transit District",
        "VCTC Valley Express",
        "City of Camarillo",
        "VCTC Intercity",
        "Western Contra Costa Transit Authority",
        "Santa Cruz Metropolitan Transit District",
        "Southern California Regional Rail Authority",
    ],

    "contactless_payments_next_six_months": [
        "City of Simi Valley",
        "Gold Coast Transit District",
        "City of Thousand Oaks",
        "City of Moorpark",
        "Yolo County Transportation District",
        "Yuba-Sutter Transit Authority",
        "City of Ojai",
        "City of Roseville",
        "Glenn County",
        "Stanislaus Regional Transit Authority",
        "South County Transit Link",
        "Golden Empire Transit District",
        "Trinity County",
        "Siskiyou County",
        "Butte County Association of Governments",
        "Imperial County Transportation Commission",
        "Madera County",
        "SunLine Transit Agency",
    ],

    "contactless_end_of_2026": [
        "Alameda-Contra Costa Transit District",
        "Anaheim Transportation Network",
        "Antelope Valley Transit Authority",
        "Peninsula Corridor Joint Powers Board",
        "Capitol Corridor Joint Powers Authority",
        "City and County of San Francisco",
        "City of Baldwin Park",
        "City of Burbank",
        "City of Carson",
        "City of Compton",
        "City of Culver City",
        "City of Fairfield",
        "City of Gardena",
        "City of Glendora",
        "City of Huntington Park",
        "City of Lawndale",
        "City of Montebello",
        "City of Monterey Park",
        "City of Morro Bay",
        "City of Norwalk",
        "City of Pasadena",
        "Redding Area Bus Authority",
        "City of Redondo Beach",
        "City of Santa Clarita",
        "City of Santa Rosa",
        "City of San Luis Obispo",
        "City of Torrance",
        "City of Union City",
        "City of Vacaville",
        "Cloverdale Transit",
        "Central Contra Costa Transit Authority",
        "Eastern Contra Costa Transit Authority",
        "El Dorado County Transit Authority",
        "Foothill Transit",
        "City of Glendale",
        "Golden Gate Bridge",
        "Humboldt Transit Authority",
        "City of Los Angeles",
        "Lake Transit Authority",
        "Livermore-Amador Valley Transit Authority",
        "Los Angeles County",
        "Los Angeles County Metropolitan Transportation Authority",
        "Los Angeles World Airports",
        "Marin County Transit District",
        "Mendocino Transit Authority",
        "Monterey-Salinas Transit",
        "Napa Valley Transportation Authority",
        "North County Transit District",
        "Nevada County",
        "Orange County Transportation Authority",
        "Palos Verdes Peninsula Transit Authority",
        "Dumbarton Bridge Regional Operations Consortium",
        "City of Petaluma",
        "Redwood Coast Transit Authority",
        "Sacramento Regional Transit District",
        "San Diego Metropolitan Transit System, Airport, Flagship Cruises",
        "San Francisco Bay Area Rapid Transit District",
        "San Mateo County Transit District",
        "Santa Clara Valley Transportation Authority",
        "City of Santa Monica",
        "Santa Barbara County Association of Governments",
        "Santa Barbara Metropolitan Transit District",
        "San Luis Obispo Regional Transit Authority",
        "Solano Transportation Authority",
        "Sonoma County Transit Schedule",
        "Sonoma-Marin Area Rail Transit District",
        "VCTC Valley Express",
        "City of Camarillo",
        "VCTC Intercity",
        "Western Contra Costa Transit Authority",
        "Santa Cruz Metropolitan Transit District",
        "Southern California Regional Rail Authority",
        "City of Simi Valley",
        "Gold Coast Transit District",
        "City of Thousand Oaks",
        "City of Moorpark",
        "Yolo County Transportation District",
        "Yuba-Sutter Transit Authority",
        "City of Roseville",
        "Glenn County",
        "Stanislaus Regional Transit Authority",
        "South County Transit Link",
        "Golden Empire Transit District",
        "Trinity Schedule",
        "Siskiyou County",
        "Butte County Association of Governments",
        "Imperial County Transportation Commission",
        "Madera County",
        "SunLine Transit Agency",
    ],

    "reduced_fares_live_now": [
        "City of Camarillo",
        "City of San Luis Obispo",
        "El Dorado County Transit Authority",
        "Gold Coast Transit District",
        "Monterey-Salinas Transit",
        "Nevada County",
        "Redding Area Bus Authority",
        "Sacramento Regional Transit District",
        "San Luis Obispo Regional Transit Authority",
        "Santa Barbara Metropolitan Transit District",
        "Santa Cruz Metro",
        "City of Simi Valley",
        "City of Thousand Oaks",
        "VCTC Valley Express",
        "Ventura County (VCTC, Gold Coast, Cities of Camarillo, Moorpark, Ojai, Simi Valley, Thousand Oaks)",
    ],

    "reduced_fares_next_6_months": [
        "Gold Coast Transit District",
        "Santa Cruz Metropolitan Transit District",
        "City of Camarillo",
        "City of Roseville",
        "VCTC Valley Express",
        "City of Simi Valley",
        "City of Thousand Oaks",
        "Santa Barbara County Association of Governments",
    ],

    "access_to_reduced_fares_contactless": [
        "Monterey-Salinas Transit",
        "Santa Barbara Metropolitan Transit District",
        "Sacramento Regional Transit District",
        "Nevada County",
        "VCTC Intercity",
        "San Luis Obispo Regional Transit Authority",
        "El Dorado County Transit Authority",
        "Redding Area Bus Authority",
        "City of San Luis Obispo",
        "Gold Coast Transit District",
        "Santa Cruz Metropolitan Transit District",
        "City of Camarillo",
        "City of Roseville",
        "VCTC Valley Express",
        "City of Simi Valley",
        "City of Thousand Oaks",
        "Santa Barbara County Association of Governments",
    ],

    # "seniors_now_ca": ["City of Camarillo"],
    # "veterans_now_ca": ["City of San Luis Obispo"],
    # "medicare_now_ca": ["El Dorado County Transit Authority"],
    # "disabilities_now_ca": ["Gold Coast Transit District"],
    # "calFresh_now_ca": [
    #     "Monterey-Salinas Transit",
    #     "Nevada County",
    #     "Redding Area Bus Authority",
    #     "Sacramento Regional Transit District",
    #     "San Luis Obispo Regional Transit Authority",
    #     "Santa Barbara Metropolitan Transit District",
    #     "Santa Cruz Metro",
    #     "City of Simi Valley",
    #     "City of Thousand Oaks",
    #     "VCTC Valley Express",
    #     "Ventura County (VCTC, Gold Coast, Cities of Camarillo, Moorpark, Ojai, Simi Valley, Thousand Oaks)",
    # ],
    # "seniors_now_ventura": ["City of Camarillo"],
    # "veterans_now_ventura": ["Gold Coast Transit District"],
    # "medicare_now_ventura": ["City of Simi Valley"],
    # "disabilities_now_ventura": ["City of Thousand Oaks"],
    # "calFresh_now_ventura": ["VCTC Valley Express", "Ventura County (VCTC, Gold Coast, Cities of Camarillo, Moorpark, Ojai, Simi Valley, Thousand Oaks)"] 
}


In [23]:
# Get unique agency names currently in the data.
analysis_names = set(orgs_stop_buffered["analysis_name"].dropna().unique())

# Get all agency names listed in the reconciliation groups.
reconciliation_names = {
    name for group in RECONCILIATION_GROUPS.values() for name in group
}

# Find reconciliation names that are missing from the data.
missing = sorted(reconciliation_names - analysis_names)

print(f"{len(missing)} names not found:\n")
for name in missing:
    print(name)

4 names not found:

Sonoma County
Sonoma County Transit Schedule
Trinity County
Trinity Schedule


In [24]:
output_folder = f"{GCS_FILE_PATH}/transit_provider_dashboard/june_2026/"

In [25]:
cols_to_weight = [
    "total_pop", "poverty_pop", "non_us_citizen",
    "workers_with_no_car", "households_with_no_cars",
    "disabled_pop", "public_asst_pop",
    "inc_extremelylow", "inc_verylow", "inc_low",
    "male_seniors", "female_seniors",
    "male_youth", "female_youth",
    "veteran_pop"
]

In [26]:
GCS__PUBLIC_FILE_PATH = f"{PUBLIC_GCS}transit_provider_dashboard/june_2026/"

### Calculating : Access to any public transit by groups.

In [27]:
route_id_shapes.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [29]:
# from matplotlib.lines import Line2D

# plot_folder = f"{GCS_FILE_PATH}/transit_provider_dashboard/maps/routes_only"
# buffered_folder = f"{plot_folder}/buffered"
# FIVE_MILES_METERS = 5 * 1609.344

# # California boundary
# ca_counties = gpd.read_file(
#     "https://www2.census.gov/geo/tiger/TIGER2024/COUNTY/tl_2024_us_county.zip"
# )
# ca_counties = ca_counties[ca_counties["STATEFP"] == "06"].to_crs(4326)
# ca_state = ca_counties.dissolve()
# xmin, ymin, xmax, ymax = ca_state.total_bounds

# # Palette
# BASE_FILL, COUNTY_LINE, STATE_EDGE = "#FAF8F4", "#B8C0CC", "#333333"
# POPULATION_AREA_COLOR, ROUTE_COLOR = "#9E4A4A", "#3F6F73"

# df_90_plot = df_90.to_crs(4326)
# orgs_stop_buffered_route_4326 = orgs_stop_buffered_route.to_crs(4326)
# route_id_shapes_4326 = route_id_shapes.to_crs(4326)

# legend_handles = [
#     Line2D([0], [0], color=POPULATION_AREA_COLOR, linewidth=1.5, label="90% population area"),
#     Line2D([0], [0], color=ROUTE_COLOR, linewidth=2, label="Routes"),
# ]

# for group_name, org_list in RECONCILIATION_GROUPS.items():
#     subset = orgs_stop_buffered_route_4326[
#         orgs_stop_buffered_route_4326["analysis_name"].isin(org_list)
#     ]

#     route_ids = (
#         subset["route_id_array"].dropna().explode().astype(str).str.strip().unique()
#     )

#     routes = gpd.clip(
#         route_id_shapes_4326[
#             route_id_shapes_4326["route_id"].astype(str).isin(route_ids)
#         ],
#         ca_state,
#     )

#     # 5-mile buffer using California State Plane meters
#     routes_buffered = (
#         routes.to_crs(3310)
#         .assign(geometry=lambda x: x.geometry.buffer(FIVE_MILES_METERS))
#         .to_crs(4326)
#     )

#     fig, ax = plt.subplots(figsize=(12, 10))

#     ca_state.plot(ax=ax, color=BASE_FILL, edgecolor=STATE_EDGE, linewidth=1, zorder=1)
#     ca_counties.boundary.plot(ax=ax, color=COUNTY_LINE, linewidth=.5, zorder=2)

#     df_90_plot.plot(
#         ax=ax, facecolor="none", edgecolor=POPULATION_AREA_COLOR,
#         linewidth=.5, zorder=3
#     )

#     # 5-mile route buffer
#     if not routes_buffered.empty:
#         routes_buffered.plot(
#             ax=ax, facecolor=ROUTE_COLOR, edgecolor=ROUTE_COLOR,
#             alpha=.25, linewidth=.5, zorder=4
#         )

#         # Original routes on top
#         routes.plot(
#             ax=ax, color=ROUTE_COLOR, alpha=.9,
#             linewidth=.8, zorder=5
#         )

#     ax.legend(
#         handles=legend_handles, loc="lower left", fontsize=8,
#         frameon=True, facecolor="white", edgecolor="none"
#     )

#     ax.set(xlim=(xmin, xmax), ylim=(ymin, ymax))
#     ax.set_title(
#         f"{group_name.replace('_', ' ').title()}\n"
#         "Area Containing 90% of CA Population, Routes, and 5-Mile Route Buffer",
#         fontsize=16,
#     )
#     ax.set_axis_off()
#     plt.tight_layout()

#     animation_path = f"{buffered_folder}/{group_name}.png"
#     with fs.open(animation_path, "wb") as f:
#         fig.savefig(f, format="png", dpi=300, bbox_inches="tight")

#     plt.close(fig)
#     print(f"Saved: {animation_path}")


In [ ]:
example_group = "contactless_payments_now_ca_msas"
plot_folder = f"{GCS_FILE_PATH}/transit_provider_dashboard/maps/routes_only"

# California boundary
ca_counties = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2024/COUNTY/tl_2024_us_county.zip")
ca_counties = ca_counties[ca_counties["STATEFP"] == "06"].to_crs(4326)
ca_state = ca_counties.dissolve()
xmin, ymin, xmax, ymax = ca_state.total_bounds

# Palette
BASE_FILL, COUNTY_LINE, STATE_EDGE = "#FAF8F4", "#B8C0CC", "#333333"
ROUTE_COLOR = "#3F6F73"

# Census tracts and population density
tracts_area = tracts_ca_acs.to_crs("EPSG:3310").copy()
tracts_area["area_km2"] = tracts_area.geometry.area / 1_000_000
tracts_area["pop_density_km2"] = tracts_area["total_pop"] / tracts_area["area_km2"]
tracts_area["pop_density"] = tracts_area["pop_density_km2"] * 2.589988
tracts_plot = tracts_area.to_crs(4326)

# Population density bins: people per square mile
density_bins = [0, 1, 10, 25, 50, 100, 250, 500, 1000, 2500, 5000, 10000, np.inf]
density_labels = ["<1", "1–10", "10–25", "25–50", "50–100", "100–250", "250–500",
                  "500–1,000", "1,000–2,500", "2,500–5,000", "5,000–10,000", "10,000+"]

density_colors = np.vstack([[1, 1, 1, 1], plt.cm.BuPu(np.linspace(.15, .90, len(density_labels) - 1))])
density_cmap = colors.ListedColormap(density_colors)
density_norm = colors.BoundaryNorm([0, 1, 10, 25, 50, 100, 250, 500, 1000, 2500, 5000, 10000, 1e9], density_cmap.N)
density_handles = [Patch(facecolor=density_colors[i], edgecolor="none", label=density_labels[i]) for i in range(len(density_labels))]
route_handle = Line2D([0], [0], color=ROUTE_COLOR, linewidth=2, label="Routes")

# Transform route data once
orgs_stop_buffered_route_4326 = orgs_stop_buffered_route.to_crs(4326)
route_id_shapes_4326 = route_id_shapes.to_crs(4326)

# Create and save maps
for group_name, org_list in RECONCILIATION_GROUPS.items():
    subset = orgs_stop_buffered_route_4326[orgs_stop_buffered_route_4326["analysis_name"].isin(org_list)].copy()
    subset = subset.explode("route_id_array")
    subset["route_id"] = subset["route_id_array"].astype(str).str.strip()
    route_pairs = subset[["feed_key", "route_id"]].dropna().drop_duplicates()

    routes = route_id_shapes_4326.merge(route_pairs, on=["feed_key", "route_id"], how="inner")
    routes = gpd.clip(routes, ca_state)

    fig, ax = plt.subplots(figsize=(12, 10))
    ca_state.plot(ax=ax, color=BASE_FILL, edgecolor=STATE_EDGE, linewidth=1, zorder=1)
    tracts_plot.plot(ax=ax, column="pop_density", cmap=density_cmap, norm=density_norm,
                     linewidth=0, alpha=0.7, zorder=2)
    ca_counties.boundary.plot(ax=ax, color=COUNTY_LINE, linewidth=.5, zorder=3)

    if not routes.empty:
        routes.plot(ax=ax, color=ROUTE_COLOR, alpha=.9, linewidth=.8, zorder=4)

    # Legend
    ax.legend(handles=density_handles + [route_handle], title="Population density\npeople / square mile",
              loc="lower left", bbox_to_anchor=(0.01, 0.12), fontsize=8, title_fontsize=9,
              frameon=True, facecolor="white", edgecolor="none")

    ax.set(xlim=(xmin, xmax), ylim=(ymin, ymax))

    # GIS-style scale bar in miles
    scale_miles, scale_deg = 100, 100 / 69.0
    scale_x, scale_y = xmin + .03 * (xmax - xmin), ymin + .035 * (ymax - ymin)
    bar_h = .008 * (ymax - ymin)

    ax.add_patch(plt.Rectangle(
        (scale_x - .015 * (xmax - xmin), scale_y - .015 * (ymax - ymin)),
        scale_deg + .03 * (xmax - xmin), bar_h + .035 * (ymax - ymin),
        facecolor="white", edgecolor="none", alpha=.9, zorder=9
    ))
    ax.add_patch(plt.Rectangle(
        (scale_x, scale_y), scale_deg, bar_h,
        facecolor="#222222", edgecolor="#222222", linewidth=1, zorder=10
    ))

    label_y = scale_y + bar_h + .006 * (ymax - ymin)
    for x, label in [(scale_x, "0"), (scale_x + scale_deg / 2, "50"), (scale_x + scale_deg, "100 miles")]:
        ax.plot([x, x], [scale_y, scale_y + bar_h + .004 * (ymax - ymin)],
                color="#222222", linewidth=1.2, zorder=11)
        ax.text(x, label_y, label, ha="center", va="bottom", fontsize=8, color="#222222", zorder=11)

    ax.set_title(f"{group_name.replace('_', ' ').title()}\nPopulation Density and Routes Serving Stops", fontsize=16)
    ax.set_axis_off()
    plt.tight_layout()

    with fs.open(f"{plot_folder}/{group_name}.png", "wb") as f:
        fig.savefig(f, format="png", dpi=300, bbox_inches="tight")

    if group_name != example_group:
        plt.close(fig)

    print(f"Saved: {plot_folder}/{group_name}.png ({len(route_pairs)} route pairs, {len(routes)} shapes)")

plt.show()


In [27]:
import numpy as np

In [ ]:
example_group = "contactless_payments_now_ca_msas"
plot_folder = f"{GCS_FILE_PATH}/transit_provider_dashboard/maps/routes_stops"

# California boundary
ca_counties = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2024/COUNTY/tl_2024_us_county.zip")
ca_counties = ca_counties[ca_counties["STATEFP"] == "06"].to_crs(4326)
ca_state = ca_counties.dissolve()
xmin, ymin, xmax, ymax = ca_state.total_bounds

# Palette
BASE_FILL, COUNTY_LINE, STATE_EDGE = "#FAF8F4", "#B8C0CC", "#333333"
STOPS_COLOR, STOPS_EDGE = "#765A91", "#5A4370"
ROUTE_COLOR = "#3F6F73"

# Census tracts and population density
tracts_area = tracts_ca_acs.to_crs("EPSG:3310").copy()
tracts_area["area_km2"] = tracts_area.geometry.area / 1_000_000
tracts_area["pop_density_km2"] = tracts_area["total_pop"] / tracts_area["area_km2"]
tracts_area["pop_density"] = tracts_area["pop_density_km2"] * 2.589988
tracts_plot = tracts_area.to_crs(4326)

# Population density bins: people per square mile
density_bins = [0, 1, 10, 25, 50, 100, 250, 500, 1000, 2500, 5000, 10000, np.inf]
density_labels = ["<1", "1–10", "10–25", "25–50", "50–100", "100–250", "250–500",
                  "500–1,000", "1,000–2,500", "2,500–5,000", "5,000–10,000", "10,000+"]

density_colors = np.vstack([[1, 1, 1, 1], plt.cm.BuPu(np.linspace(.15, .90, len(density_labels) - 1))])
density_cmap = colors.ListedColormap(density_colors)
density_norm = colors.BoundaryNorm([0, 1, 10, 25, 50, 100, 250, 500, 1000, 2500, 5000, 10000, 1e9], density_cmap.N)
density_handles = [Patch(facecolor=density_colors[i], edgecolor="none", label=density_labels[i]) for i in range(len(density_labels))]

stop_handle = Line2D([0], [0], marker="o", color="none", markerfacecolor=STOPS_COLOR,
                     markeredgecolor=STOPS_EDGE, markersize=8, label="Buffered stops")
route_handle = Line2D([0], [0], color=ROUTE_COLOR, linewidth=2, label="Routes")

# Transform once
orgs_stop_buffered_route_4326 = orgs_stop_buffered_route.to_crs(4326)
route_id_shapes_4326 = route_id_shapes.to_crs(4326)

for group_name, org_list in RECONCILIATION_GROUPS.items():
    subset = orgs_stop_buffered_route_4326[
        orgs_stop_buffered_route_4326["analysis_name"].isin(org_list)
    ].copy()

    # One row per feed_key + route_id
    route_pairs = (
        subset[["feed_key", "route_id_array"]]
        .explode("route_id_array")
        .rename(columns={"route_id_array": "route_id"})
        .dropna(subset=["feed_key", "route_id"])
    )

    route_pairs["feed_key"] = route_pairs["feed_key"].astype(str).str.strip()
    route_pairs["route_id"] = route_pairs["route_id"].astype(str).str.strip()
    route_pairs = route_pairs.drop_duplicates()

    # Match route shapes using BOTH feed_key and route_id
    shape_routes = route_id_shapes_4326.copy()
    shape_routes["feed_key"] = shape_routes["feed_key"].astype(str).str.strip()
    shape_routes["route_id"] = shape_routes["route_id"].astype(str).str.strip()

    routes = shape_routes.merge(route_pairs, on=["feed_key", "route_id"], how="inner")
    routes = gpd.clip(routes, ca_state)

    print(group_name, "| stops:", len(subset), "| route pairs:", len(route_pairs), "| route shapes:", len(routes))

    fig, ax = plt.subplots(figsize=(12, 10))

    # California background
    ca_state.plot(ax=ax, color=BASE_FILL, edgecolor=STATE_EDGE, linewidth=1, zorder=1)

    # Population density
    tracts_plot.plot(ax=ax, column="pop_density", cmap=density_cmap, norm=density_norm,
                     linewidth=0, alpha=0.7, zorder=2)

    # County boundaries
    ca_counties.boundary.plot(ax=ax, color=COUNTY_LINE, linewidth=.5, zorder=3)

    # Buffered stops
    if not subset.empty:
        subset.plot(ax=ax, facecolor=STOPS_COLOR, alpha=.9, edgecolor=STOPS_EDGE, linewidth=.3, zorder=10)

    # Correct routes matched by feed_key + route_id
    if not routes.empty:
        routes.plot(ax=ax, color=ROUTE_COLOR, alpha=.9, linewidth=1.2, zorder=9)

    # Legend
    ax.legend(handles=density_handles + [stop_handle, route_handle],
              title="Population density\npeople / square mile",
              loc="lower left", bbox_to_anchor=(0.01, 0.12),
              fontsize=8, title_fontsize=9, frameon=True,
              facecolor="white", edgecolor="none")

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    # GIS-style scale bar in miles
    scale_miles, scale_deg = 100, 100 / 69.0
    scale_x, scale_y = xmin + .03 * (xmax - xmin), ymin + .035 * (ymax - ymin)
    bar_h = .008 * (ymax - ymin)

    ax.add_patch(plt.Rectangle(
        (scale_x - .015 * (xmax - xmin), scale_y - .015 * (ymax - ymin)),
        scale_deg + .03 * (xmax - xmin), bar_h + .035 * (ymax - ymin),
        facecolor="white", edgecolor="none", alpha=.9, zorder=9
    ))
    ax.add_patch(plt.Rectangle(
        (scale_x, scale_y), scale_deg, bar_h,
        facecolor="#222222", edgecolor="#222222", linewidth=1, zorder=10
    ))

    label_y = scale_y + bar_h + .006 * (ymax - ymin)
    for x, label in [(scale_x, "0"), (scale_x + scale_deg / 2, "50"), (scale_x + scale_deg, "100 miles")]:
        ax.plot([x, x], [scale_y, scale_y + bar_h + .004 * (ymax - ymin)],
                color="#222222", linewidth=1.2, zorder=11)
        ax.text(x, label_y, label, ha="center", va="bottom", fontsize=8, color="#222222", zorder=11)

    ax.set_title(f"{group_name.replace('_', ' ').title()}\n"
                 "Population Density and Routes Serving Stops, Buffered Stops, and Routes", fontsize=16)

    ax.set_axis_off()
    plt.tight_layout()

    with fs.open(f"{plot_folder}/{group_name}.png", "wb") as f:
        fig.savefig(f, format="png", dpi=300, bbox_inches="tight")

    if group_name != example_group:
        plt.close(fig)

    print(f"Saved: {plot_folder}/{group_name}.png")

plt.show()


In [28]:
ca_counties = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2024/COUNTY/tl_2024_us_county.zip")
ca_counties = ca_counties[ca_counties["STATEFP"] == "06"].to_crs(4326)
ca_state = ca_counties.dissolve()
xmin, ymin, xmax, ymax = ca_state.total_bounds

In [ ]:
from matplotlib.patches import Patch

example_group = "reduced_fares_next_6_months"
animation_folder = f"{GCS_FILE_PATH}/transit_provider_dashboard/maps/animation/routes/"
FIVE_MILES_METERS = 5 * 1609.344

BASE_FILL, COUNTY_LINE, STATE_EDGE = "#FAF8F4", "#B8C0CC", "#333333"
ROUTE_COLOR = "#3F6F73"

ca_counties = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2024/COUNTY/tl_2024_us_county.zip"
)
ca_counties = ca_counties[ca_counties["STATEFP"] == "06"].to_crs(4326)
ca_state = ca_counties.dissolve()
xmin, ymin, xmax, ymax = ca_state.total_bounds

# Census tracts and population density
tracts_plot = tracts_ca_acs.to_crs("EPSG:3310").copy()
tracts_plot["area_km2"] = tracts_plot.geometry.area / 1_000_000
tracts_plot["pop_density"] = (
    tracts_plot["total_pop"] / tracts_plot["area_km2"]
) * 2.589988
tracts_plot = tracts_plot.to_crs(4326)

# Population density bins: people per square mile
density_bins = [0, 1, 10, 25, 50, 100, 250, 500, 1000, 2500, 5000, 10000, np.inf]
density_labels = [
    "<1",
    "1–10",
    "10–25",
    "25–50",
    "50–100",
    "100–250",
    "250–500",
    "500–1,000",
    "1,000–2,500",
    "2,500–5,000",
    "5,000–10,000",
    "10,000+",
]

density_colors = np.vstack([
    [1, 1, 1, 1],
    plt.cm.BuPu(np.linspace(.15, .90, len(density_labels) - 1))
])

density_cmap = colors.ListedColormap(density_colors)

density_norm = colors.BoundaryNorm(
    [0, 1, 10, 25, 50, 100, 250, 500, 1000, 2500, 5000, 10000, 1e9],
    density_cmap.N
)

# Legend handles for population density
density_handles = [
    Patch(
        facecolor=density_colors[i],
        edgecolor="none",
        label=density_labels[i]
    )
    for i in range(len(density_labels))
]

orgs_stop_buffered_route_4326 = orgs_stop_buffered_route.to_crs(4326)
route_id_shapes_4326 = route_id_shapes.to_crs(4326)

# Clean route-shape keys once
shape_routes = route_id_shapes_4326.copy()
shape_routes["feed_key"] = shape_routes["feed_key"].astype(str).str.strip()
shape_routes["route_id"] = shape_routes["route_id"].astype(str).str.strip()

for group_name, org_list in RECONCILIATION_GROUPS.items():

    subset = orgs_stop_buffered_route_4326[
        orgs_stop_buffered_route_4326["analysis_name"].isin(org_list)
    ].copy()

    # Get feed_key + route_id pairs for this reconciliation group
    route_pairs = (
        subset[["feed_key", "route_id_array"]]
        .explode("route_id_array")
        .rename(columns={"route_id_array": "route_id"})
        .dropna(subset=["feed_key", "route_id"])
    )

    route_pairs["feed_key"] = (
        route_pairs["feed_key"].astype(str).str.strip()
    )
    route_pairs["route_id"] = (
        route_pairs["route_id"].astype(str).str.strip()
    )

    route_pairs = route_pairs.drop_duplicates()

    # Match route shapes using BOTH feed_key and route_id
    routes = gpd.clip(
        shape_routes.merge(
            route_pairs,
            on=["feed_key", "route_id"],
            how="inner"
        ),
        ca_state
    ).copy()

    # 5-mile route buffer
    routes_buffered_4326 = (
        routes
        .to_crs(3310)
        .assign(
            geometry=lambda x: x.geometry.buffer(FIVE_MILES_METERS)
        )
        .to_crs(4326)
    )

    fig, ax = plt.subplots(figsize=(12, 10))

    # Leave room underneath the map for legend + scale bar
    fig.subplots_adjust(bottom=.24)

    def draw_base():
        ca_state.plot(
            ax=ax,
            color=BASE_FILL,
            edgecolor=STATE_EDGE,
            linewidth=1,
            zorder=1
        )

        tracts_plot.plot(
            ax=ax,
            column="pop_density",
            cmap=density_cmap,
            norm=density_norm,
            linewidth=0,
            alpha=.7,
            zorder=2
        )

        ca_counties.boundary.plot(
            ax=ax,
            color=COUNTY_LINE,
            linewidth=.5,
            zorder=3
        )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_axis_off()

    def draw_legend():
        ax.legend(
            handles=density_handles,
            title="Population density\npeople / square mile",
            loc="upper left",
            bbox_to_anchor=(0, -.03),
            fontsize=6.5,
            title_fontsize=7.5,
            frameon=True,
            facecolor="white",
            edgecolor="#CCCCCC",
            framealpha=.95,
            labelspacing=.3,
            handlelength=1.2,
            borderpad=.6,
            ncol=2
        )

    def draw_scale():
        # Scale bar positioned below the map using axes coordinates
        x0, y0 = .55, -.17
        w, h = .25, .018

        # Black half
        ax.add_patch(
            plt.Rectangle(
                (x0, y0),
                w / 2,
                h,
                transform=ax.transAxes,
                facecolor="#222222",
                edgecolor="#222222",
                clip_on=False,
                zorder=10
            )
        )

        # White half
        ax.add_patch(
            plt.Rectangle(
                (x0 + w / 2, y0),
                w / 2,
                h,
                transform=ax.transAxes,
                facecolor="white",
                edgecolor="#222222",
                clip_on=False,
                zorder=10
            )
        )

        # Tick marks + labels
        for x, label in [
            (x0, "0"),
            (x0 + w / 2, "100"),
            (x0 + w, "200 miles")
        ]:
            ax.plot(
                [x, x],
                [y0, y0 + h + .012],
                transform=ax.transAxes,
                color="#222222",
                linewidth=1.2,
                clip_on=False,
                zorder=11
            )

            ax.text(
                x,
                y0 + h + .015,
                label,
                transform=ax.transAxes,
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="#222222",
                clip_on=False,
                zorder=11
            )

    def update(frame):
        ax.clear()
        draw_base()

        alpha = max(0, min(1, (frame - 30) / 30))
        title = "Before" if alpha == 0 else "After"

        if alpha > 0 and not routes.empty:

            # 5-mile route buffer
            routes_buffered_4326.plot(
                ax=ax,
                facecolor=ROUTE_COLOR,
                edgecolor=ROUTE_COLOR,
                alpha=.25 * alpha,
                linewidth=.5,
                zorder=4
            )

            # Route shapes
            routes.plot(
                ax=ax,
                color=ROUTE_COLOR,
                alpha=.9 * alpha,
                linewidth=.8,
                zorder=5
            )

        # These must be redrawn every frame because ax.clear()
        # removes the previous legend and scale bar.
        draw_legend()
        draw_scale()

        ax.set_title(
            f"{title} — {group_name.replace('_', ' ').title()}\n"
            "Population Density + Routes and 5-Mile Route Buffer",
            fontsize=16,
            color="#222222"
        )

        return []

    update(0)
    fig.canvas.draw()

    anim = FuncAnimation(
        fig,
        update,
        frames=75,
        interval=60,
        repeat=True
    )

    html = anim.to_jshtml()

    animation_path = (
        f"{animation_folder}/{group_name}_before_after.html"
    )

    with fs.open(animation_path, "w") as f:
        f.write(html)

    if group_name == example_group:
        display(HTML(html))

    plt.close(fig)

    print(
        f"Saved: {animation_path} | "
        f"{len(route_pairs)} route pairs | "
        f"{len(routes)} shapes"
    )


In [30]:
df_90.head(2)

,geometry,n_block_groups,total_population,total_land_area_sqmi
0,"MULTIPOLYGON (((-120.71444 37.39681, -120.7230...",22506,35357649.0,6614.833217


In [34]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

In [ ]:
from matplotlib.patches import Patch

example_group = "reduced_fares_next_6_months"
animation_folder = f"{GCS_FILE_PATH}/transit_provider_dashboard/maps/animation/routes/"
FIVE_MILES_METERS = 5 * 1609.344

BASE_FILL, COUNTY_LINE, STATE_EDGE = "#FAF8F4", "#B8C0CC", "#333333"
ROUTE_COLOR = "#3F6F73"

ca_counties = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2024/COUNTY/tl_2024_us_county.zip")
ca_counties = ca_counties[ca_counties["STATEFP"] == "06"].to_crs(4326)
ca_state = ca_counties.dissolve()
xmin, ymin, xmax, ymax = ca_state.total_bounds

# Census tracts + population density
tracts_plot = tracts_ca_acs.to_crs(3310).copy()
tracts_plot["area_km2"] = tracts_plot.geometry.area / 1_000_000
tracts_plot["pop_density"] = (tracts_plot["total_pop"] / tracts_plot["area_km2"]) * 2.589988
tracts_plot = tracts_plot.to_crs(4326)

density_labels = ["<1", "1–10", "10–25", "25–50", "50–100", "100–250", "250–500",
                  "500–1,000", "1,000–2,500", "2,500–5,000", "5,000–10,000", "10,000+"]
density_colors = np.vstack([[1, 1, 1, 1], plt.cm.BuPu(np.linspace(.15, .90, len(density_labels) - 1))])
density_cmap = colors.ListedColormap(density_colors)
density_norm = colors.BoundaryNorm(
    [0, 1, 10, 25, 50, 100, 250, 500, 1000, 2500, 5000, 10000, 1e9],
    density_cmap.N
)
density_handles = [Patch(facecolor=density_colors[i], edgecolor="none", label=density_labels[i])
                   for i in range(len(density_labels))]

orgs_stop_buffered_route_4326 = orgs_stop_buffered_route.to_crs(4326)
shape_routes = route_id_shapes.to_crs(4326).copy()
shape_routes["feed_key"] = shape_routes["feed_key"].astype(str).str.strip()
shape_routes["route_id"] = shape_routes["route_id"].astype(str).str.strip()

for group_name, org_list in RECONCILIATION_GROUPS.items():
    subset = orgs_stop_buffered_route_4326[
        orgs_stop_buffered_route_4326["analysis_name"].isin(org_list)
    ].copy()

    route_pairs = (subset[["feed_key", "route_id_array"]].explode("route_id_array")
                   .rename(columns={"route_id_array": "route_id"})
                   .dropna(subset=["feed_key", "route_id"]))
    route_pairs["feed_key"] = route_pairs["feed_key"].astype(str).str.strip()
    route_pairs["route_id"] = route_pairs["route_id"].astype(str).str.strip()
    route_pairs = route_pairs.drop_duplicates()

    routes = gpd.clip(
        shape_routes.merge(route_pairs, on=["feed_key", "route_id"], how="inner"),
        ca_state
    ).copy()

    routes_buffered_4326 = (
        routes.to_crs(3310)
        .assign(geometry=lambda x: x.geometry.buffer(FIVE_MILES_METERS))
        .to_crs(4326)
    )

    fig, ax = plt.subplots(figsize=(12, 10))
    fig.subplots_adjust(bottom=.24)

    def draw_base():
        ca_state.plot(ax=ax, color=BASE_FILL, edgecolor=STATE_EDGE, linewidth=1, zorder=1)
        tracts_plot.plot(ax=ax, column="pop_density", cmap=density_cmap, norm=density_norm,
                         linewidth=0, alpha=.7, zorder=2)
        ca_counties.boundary.plot(ax=ax, color=COUNTY_LINE, linewidth=.5, zorder=3)
        ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax); ax.set_axis_off()

    def draw_legend():
        ax.legend(handles=density_handles, title="Population density\npeople / square mile",
                  loc="upper left", bbox_to_anchor=(0, -.03), fontsize=6.5, title_fontsize=7.5,
                  frameon=True, facecolor="white", edgecolor="#CCCCCC", framealpha=.95,
                  labelspacing=.3, handlelength=1.2, borderpad=.6, ncol=2)

    def draw_scale():
        x0, y0, w, h = .55, -.17, .25, .018
        for x, fc in [(x0, "#222222"), (x0 + w / 2, "white")]:
            ax.add_patch(plt.Rectangle((x, y0), w / 2, h, transform=ax.transAxes,
                                       facecolor=fc, edgecolor="#222222",
                                       clip_on=False, zorder=10))
        for x, label in [(x0, "0"), (x0 + w / 2, "100"), (x0 + w, "200 miles")]:
            ax.plot([x, x], [y0, y0 + h + .012], transform=ax.transAxes,
                    color="#222222", linewidth=1.2, clip_on=False, zorder=11)
            ax.text(x, y0 + h + .015, label, transform=ax.transAxes,
                    ha="center", va="bottom", fontsize=7.5, color="#222222",
                    clip_on=False, zorder=11)

    def update(frame):
        ax.clear(); draw_base()
        alpha = max(0, min(1, (frame - 30) / 30))
        title = "Before" if alpha == 0 else "After"

        if alpha > 0 and not routes.empty:
            routes_buffered_4326.plot(ax=ax, facecolor=ROUTE_COLOR, edgecolor=ROUTE_COLOR,
                                      alpha=.25 * alpha, linewidth=.5, zorder=4)
            routes.plot(ax=ax, color=ROUTE_COLOR, alpha=.9 * alpha, linewidth=.8, zorder=5)

        draw_legend(); draw_scale()
        ax.set_title(
            f"{title} — {group_name.replace('_', ' ').title()}\n"
            "Population Density + Routes and 5-Mile Route Buffer",
            fontsize=16, color="#222222"
        )
        return []

    update(0); fig.canvas.draw()
    anim = FuncAnimation(fig, update, frames=75, interval=60, repeat=True)
    html = anim.to_jshtml()
    animation_path = f"{animation_folder}/{group_name}_before_after.html"

    with fs.open(animation_path, "w") as f:
        f.write(html)

    if group_name == example_group:
        display(HTML(html))

    plt.close(fig)
    print(f"Saved: {animation_path} | {len(route_pairs)} route pairs | {len(routes)} shapes")
